# 02 · Train the compact Transformer on TRAIN and VALIDATION

Use the identity-preserving bundle and frozen patient-level split created or verified in notebook 01. This notebook independently reloads those artifacts so it can run in a fresh kernel. It does not require notebook 01's Python variables.

**Inputs:** the original raw-count sequence bundle, complete frozen split manifest and local configuration. **Outputs:** best checkpoint, epoch history, training metadata and aggregate validation summaries. No TEST inference or performance evaluation occurs here.

The shared reference specifies the intended sequential representation and proposed model; it does not establish a saved checkpoint or completed training. This notebook is a concrete downstream implementation to run on the work laptop. No real model has been trained by merely downloading it. LightGBM remains deferred.

## 1. Load the same local configuration and dependencies

Install the organization's approved CPU/GPU PyTorch build, then `requirements-training.txt`. Keep the configuration from notebook 01 and choose a new or empty `run_dir`. A nonempty run directory fails instead of overwriting an existing experiment. Use a fresh directory for each validation-driven experiment; freeze the selected run before TEST evaluation.

`TAK861_CONFIG` can point to the local JSON file. Without it, the loader uses the repository's ignored `config.local.json`. Artifact paths resolve relative to that configuration file.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "targeting_evaluation.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Launch this notebook from the repository directory or one of its child directories.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.configuration import load_config

cfg = load_config()

In [ ]:
import matplotlib.pyplot as plt

from src.data_utils import load_bundle, split_indices, validate_population, validate_source_review
from src.model import ModelConfig
from src.training import TrainConfig, train_transformer
from targeting_evaluation import read_input, validate_manifest

validate_source_review(cfg["source_review"])
bundle = load_bundle(cfg["bundle_dir"])
validate_population(bundle, cfg["expected"])
manifest = validate_manifest(read_input(cfg["manifest_path"]))
indices = split_indices(bundle, manifest)
assert sum(len(rows) for rows in indices.values()) == len(bundle.y)

## 2. Review the configured model and training choices

The default model projects each 1,028-channel month to 128 dimensions, adds learned temporal positions and applies two Transformer encoder layers with four attention heads, a 256-dimensional feedforward block and dropout 0.2. Layer normalization and mean pooling across the 12 historical months feed a small binary-logit head. The time and feature dimensions are derived from the verified bundle.

Zero-activity months are valid observations and remain in pooling; they are not padding. All included months must precede the reviewed prediction cutoff. An attention mask is not used to conceal leaked events; upstream cutoff validation must establish their exclusion.

Default preprocessing is per-batch `log1p` of raw nonnegative counts, applied exactly once. It fits no statistics on VALIDATION or TEST. TRAIN's negative-to-positive ratio determines the positive weight in `BCEWithLogitsLoss`. All TRAIN snapshots are retained. AdamW, gradient clipping and early stopping limit the initial experiment's size.

In [ ]:
model_cfg = ModelConfig(
    input_dim=bundle.X.shape[2],
    seq_len=bundle.X.shape[1],
    **cfg["model"],
)
train_cfg = TrainConfig(**cfg["training"])

from dataclasses import asdict
display(pd.DataFrame([asdict(model_cfg)]))
display(pd.DataFrame([asdict(train_cfg)]))

## 3. Train and select the checkpoint using VALIDATION only

Early stopping monitors VALIDATION average precision (non-interpolated AP). The highest observed AP is checkpointed; exact AP ties retain the earlier epoch. `min_delta` controls the patience counter, while checkpointing still retains the highest observed AP.

After restoring that checkpoint, a single threshold is selected by maximum VALIDATION F1; exact ties select the highest threshold. The checkpoint stores this threshold, preprocessing, feature/time order and fingerprints of the complete bundle and frozen manifest. These fingerprints bind the artifacts; they are not TEST performance statistics.

Training uses only TRAIN batches and VALIDATION batches. TEST keys/labels participate in the full-artifact integrity checks, but no TEST features are used in a gradient update and no TEST scores, loss or metrics guide model selection. The device setting can be `auto`, `cpu` or the approved available CUDA device. Default `num_workers=0` avoids multiprocessing assumptions in notebook and Windows environments.

In [ ]:
training_result = train_transformer(
    bundle=bundle,
    manifest=manifest,
    output_dir=cfg["run_dir"],
    model_config=model_cfg,
    train_config=train_cfg,
)
training_summary = training_result["summary"]
assert training_summary["test_inference_performed"] is False
assert training_summary["training_split"] == "TRAIN"
assert training_summary["tuning_split"] == "VALIDATION"

## 4. Inspect aggregate training and validation results

Loss uses TRAIN-derived positive weighting, so treat it as an optimization diagnostic. Validation AP is the checkpoint-selection metric. The displayed validation operating metrics reuse the threshold selected on those validation scores and are therefore selection-set summaries; the later frozen TEST run supplies the held-out assessment.

No patient identifiers, per-snapshot predictions or feature-name lists are displayed.

In [ ]:
summary_fields = [
    "model_parameter_count", "epochs_completed", "best_epoch",
    "best_validation_average_precision", "train_pos_weight",
    "validation_threshold", "resolved_device", "test_inference_performed",
]
display(pd.DataFrame([{name: training_summary[name] for name in summary_fields}]))
display(pd.DataFrame([training_summary["validation_metrics"]]))

history = pd.read_csv(training_result["history_path"])
display(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(history["epoch"], history["training_loss"], label="TRAIN loss")
axes[0].plot(history["epoch"], history["validation_loss"], label="VALIDATION loss")
axes[0].set(xlabel="Epoch", ylabel="Weighted binary cross-entropy", title="Training diagnostics")
axes[0].legend()
axes[1].plot(history["epoch"], history["validation_average_precision"], marker="o", label="VALIDATION AP")
axes[1].axvline(training_summary["best_epoch"], color="gray", linestyle="--", label="Saved checkpoint")
axes[1].set(xlabel="Epoch", ylabel="Average precision", title="Checkpoint selection")
axes[1].legend()
fig.savefig(cfg["run_dir"] / "training_history.png", dpi=160)
plt.show()
plt.close(fig)

## 5. Freeze the selected run, then evaluate TEST

The run directory contains `best_transformer.pt`, `training_history.csv`, `training_metadata.json` and the aggregate training-history chart. These are local artifacts and remain outside Git. A failed/interrupted run must not be treated as a completed checkpoint; the evaluation loader requires the completion flag and frozen validation threshold.

Use **03_transformer_evaluation.ipynb** only after choosing the final configuration/run using TRAIN and VALIDATION. It will verify artifact fingerprints, score the frozen TEST snapshots, apply the saved threshold and report detailed decile/gains/lift metrics. Do not choose another model or operating threshold because it looks better on those TEST results.

Positive-class weighting can change probability calibration. Sigmoid outputs provide the ranking scores used for targeting; this training run does not establish calibrated clinical probabilities. The initial model is a defensible starting point, not evidence of superiority over a deferred LightGBM benchmark.